## Notebook 06 — Numeric Feature Correlation Analysis

---

This notebook:
1. Loads + merges data, drops >80% missing columns, removes `TransactionID`
2. Performs stratified 80/20 train-validation split
3. Selects only numeric columns from X_train
4. Computes correlation with y_train (point-biserial/Pearson)
5. Computes absolute correlation
6. Sorts by absolute correlation (descending)
7. Prints top 20 numeric features

> **No data modification or preprocessing is performed.**

---
## 📦 Step 0 — Import Libraries, Load & Split Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'
TARGET    = 'isFraud'

# Load & merge
print('Loading and merging...')
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

# Drop >80% missing columns
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw

# Drop identifier column
train = train.drop(columns=['TransactionID'], errors='ignore')

# Stratified 80/20 split
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
del train, X, y

print(f'✅ X_train shape : {X_train.shape}')
print(f'✅ X_val shape   : {X_val.shape}')
print(f'✅ Train fraud % : {y_train.mean()*100:.2f}%')
print(f'✅ Val fraud %   : {y_val.mean()*100:.2f}%')

Loading and merging...
✅ X_train shape : (472432, 358)
✅ X_val shape   : (118108, 358)
✅ Train fraud % : 3.50%
✅ Val fraud %   : 3.50%


---
## 🔢 Step 1 — Select Numeric Columns Only

In [2]:
# Select only numeric columns (int, float)
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
X_train_numeric = X_train[numeric_cols]

print(f'✅ Total columns in X_train: {X_train.shape[1]}')
print(f'✅ Numeric columns: {len(numeric_cols)}')
print(f'✅ Non-numeric columns: {X_train.shape[1] - len(numeric_cols)}')

✅ Total columns in X_train: 358
✅ Numeric columns: 332
✅ Non-numeric columns: 26


---
## 📊 Step 2 — Compute Correlation with Target (y_train)

In [3]:
# Compute correlation for each numeric column with y_train
correlations = []

for col in numeric_cols:
    # Handle columns with missing values by using pairwise complete observations
    valid_mask = X_train_numeric[col].notna()
    
    if valid_mask.sum() < 2:
        # Skip columns with less than 2 valid values
        continue
    
    # Compute correlation (Pearson / point-biserial)
    corr = X_train_numeric.loc[valid_mask, col].corr(y_train[valid_mask])
    
    # Handle NaN correlations (e.g., constant columns)
    if pd.isna(corr):
        corr = 0.0
    
    correlations.append({
        'Column': col,
        'Correlation': corr,
        'Abs Correlation': abs(corr)
    })

# Create DataFrame and sort by absolute correlation
corr_df = pd.DataFrame(correlations).sort_values('Abs Correlation', ascending=False).reset_index(drop=True)
corr_df.index += 1

print(f'✅ Computed correlations for {len(corr_df)} numeric features')

✅ Computed correlations for 332 numeric features


---
## 🏆 Step 3 — Top 20 Numeric Features by Absolute Correlation

In [4]:
top20 = corr_df.head(20)

W    = 70
sep  = '=' * W
dash = '-' * W
hdr  = f'{"Rank":<5} {"Column":<25} {"Correlation":>18} {"Abs Correlation":>18}'

print(sep)
print('   TOP 20 Numeric Features — Sorted by Absolute Correlation')
print(sep)
print(hdr)
print(dash)
for rank, row in top20.iterrows():
    print(
        f'{rank:<5} {row["Column"]:<25} '
        f'{row["Correlation"]:>18.5f} '
        f'{row["Abs Correlation"]:>18.5f}'
    )
print(sep)

   TOP 20 Numeric Features — Sorted by Absolute Correlation
Rank  Column                           Correlation    Abs Correlation
----------------------------------------------------------------------
1     V257                                 0.38217            0.38217
2     V246                                 0.36709            0.36709
3     V244                                 0.36557            0.36557
4     V242                                 0.36228            0.36228
5     V201                                 0.32732            0.32732
6     V200                                 0.31810            0.31810
7     V189                                 0.30964            0.30964
8     V188                                 0.30419            0.30419
9     V258                                 0.30087            0.30087
10    V45                                  0.28055            0.28055
11    V228                                 0.26850            0.26850
12    V44                    